In [ ]:
!pip install geopandas hdbscan xgboost

# Tahap 1 Setup, import library, dan baca data

In [ ]:
import os, joblib
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import hdbscan
from shapely.geometry import Point
from scipy.stats import randint, uniform, loguniform
from sklearn.neighbors import BallTree
from sklearn.metrics.pairwise import haversine_distances
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, precision_recall_curve,
                             confusion_matrix, roc_curve, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

# ---------------- konfigurasi ----------------
BATAS_TRAIN = "2024-12-31"
RS = 42
STASIUN = {                      # verifikasi koordinat ini dengan data NASA POWER
    "Palangka Raya": (-2.21, 113.92),
    "Sampit":        (-2.53, 112.95),
    "Pangkalan Bun": (-2.68, 111.62),
    "Muara Teweh":   (-0.95, 114.89),
    "Kuala Kapuas":  (-3.00, 114.39),
}
VAR = ["T2M", "RH2M", "PRECTOTCORR", "WS10M"]

# ---------------- cache ----------------
PAKAI_DRIVE = True      # False kalau tidak mau menyentuh Google Drive
if PAKAI_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/hotspot_kalteng"
else:
    SAVE_DIR = "/content/cache"
os.makedirs(SAVE_DIR, exist_ok=True)

def cached(nama, fn):
    """Kalau file cache ada -> load; kalau belum -> hitung, simpan, lalu kembalikan."""
    path = f"{SAVE_DIR}/{nama}.joblib"
    if os.path.exists(path):
        print("[cache] load:", nama)
        return joblib.load(path)
    hasil = fn()
    joblib.dump(hasil, path)
    print("[cache] simpan:", nama)
    return hasil

# ---------------- baca data ----------------
df_hotspot = pd.read_csv("/content/hotspot_kalimantan_tengah.csv")
df_meteo   = pd.read_csv("/content/meteorologi_kalimantan_tengah.csv")
batas      = gpd.read_file("/content/Kalimantan Tengah.geojson")
print("Ukuran dataset hotspot:", df_hotspot.shape)
print("Ukuran dataset meteorologi:", df_meteo.shape)

# Tahap 2 Cleaning data meteorologi

In [ ]:
# salin dari df_meteo dan ubah kolom tanggal ke datetime
met = df_meteo.copy()
met["tanggal"] = pd.to_datetime(met["tanggal"])

met = met.drop_duplicates(["lokasi", "tanggal"]).sort_values(["lokasi", "tanggal"])

# -999 = missing value NASA POWER
met[VAR] = met[VAR].replace(-999, np.nan)
print("Missing sebelum diisi:\n", met[VAR].isna().sum())

# isi missing dengan interpolasi per stasiun
met[VAR] = met.groupby("lokasi")[VAR].transform(
    lambda s: s.interpolate(limit_direction="both"))
print("Missing sesudah diisi:", met[VAR].isna().sum().sum())

# fitur turunan (rolling per stasiun)
g = met.groupby("lokasi")
met["CH_3d"]  = g["PRECTOTCORR"].transform(lambda s: s.rolling(3,  min_periods=1).sum())
met["CH_7d"]  = g["PRECTOTCORR"].transform(lambda s: s.rolling(7,  min_periods=1).sum())
met["CH_14d"] = g["PRECTOTCORR"].transform(lambda s: s.rolling(14, min_periods=1).sum())
met["RH_7d"]  = g["RH2M"].transform(lambda s: s.rolling(7, min_periods=1).mean())
met["T_7d"]   = g["T2M"].transform(lambda s: s.rolling(7, min_periods=1).mean())

def hari_kering_berturut(s, ambang=1.0):
    kering = s < ambang
    grup = (~kering).cumsum()
    return kering.groupby(grup).cumsum()

met["HKB"] = g["PRECTOTCORR"].transform(hari_kering_berturut)  # hari kering berturut-turut

# Tahap 3 Cleaning data hotspot dan pemotongan batas provinsi

In [ ]:
hs = df_hotspot.copy()
hs["acq_date"] = pd.to_datetime(hs["acq_date"])
print("Jumlah awal:", len(hs))

# buang deteksi confidence rendah ('l'); hapus baris ini kalau mau semua dipakai
hs = hs[hs["confidence"].isin(["n", "h"])]
hs = hs.rename(columns={"acq_date": "tanggal"})
hs = hs[["tanggal", "latitude", "longitude"]].dropna().reset_index(drop=True)
print("Setelah filter confidence:", len(hs))

# ---- potong ke batas provinsi ----
batas = batas.to_crs("EPSG:4326")
try:
    batas_union = batas.geometry.union_all()      # geopandas >= 1.0
except AttributeError:
    batas_union = batas.geometry.unary_union      # geopandas lama

gdf_hs = gpd.GeoDataFrame(
    hs.copy(),
    geometry=gpd.points_from_xy(hs["longitude"], hs["latitude"]),
    crs="EPSG:4326",
)
dalam = gdf_hs.geometry.intersects(batas_union)
print("Hotspot dalam provinsi :", dalam.sum())
print("Hotspot di luar        :", (~dalam).sum())

hs = gdf_hs.loc[dalam].drop(columns="geometry").reset_index(drop=True)

# ---- cek posisi stasiun terhadap batas provinsi ----
for nm, (la, lo) in STASIUN.items():
    print(f"{nm:15s} di dalam provinsi: {batas_union.contains(Point(lo, la))}")

# Tahap 3 Cleaning data hotspot dan pemotongan batas provinsi

In [ ]:
hs = df_hotspot.copy()
hs["acq_date"] = pd.to_datetime(hs["acq_date"])
print("Jumlah awal:", len(hs))

# buang deteksi confidence rendah ('l'); hapus baris ini kalau mau semua dipakai
hs = hs[hs["confidence"].isin(["n", "h"])]
hs = hs.rename(columns={"acq_date": "tanggal"})
hs = hs[["tanggal", "latitude", "longitude"]].dropna().reset_index(drop=True)
print("Setelah filter confidence:", len(hs))

# ---- potong ke batas provinsi ----
batas = batas.to_crs("EPSG:4326")
try:
    batas_union = batas.geometry.union_all()      # geopandas >= 1.0
except AttributeError:
    batas_union = batas.geometry.unary_union      # geopandas lama

gdf_hs = gpd.GeoDataFrame(
    hs.copy(),
    geometry=gpd.points_from_xy(hs["longitude"], hs["latitude"]),
    crs="EPSG:4326",
)
dalam = gdf_hs.geometry.intersects(batas_union)
print("Hotspot dalam provinsi :", dalam.sum())
print("Hotspot di luar        :", (~dalam).sum())

hs = gdf_hs.loc[dalam].drop(columns="geometry").reset_index(drop=True)

# ---- cek posisi stasiun terhadap batas provinsi ----
for nm, (la, lo) in STASIUN.items():
    print(f"{nm:15s} di dalam provinsi: {batas_union.contains(Point(lo, la))}")